# DataInspector - Modular Data Sanitization & Exploration Engine



## Python Package Installation & Imports

In [ ]:
import json
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
import warnings
from google.colab import files
from IPython.display import display, HTML

warnings.filterwarnings('ignore')

## Core Classes Definition

In [ ]:
class PlottingMethods:
    """Modular plotting utilities."""

    @staticmethod
    def get_methods_info():
        """Return available plotting methods."""
        return {
            'response': {
                'methods': ['plot_bar_chart', 'plot_pie_chart', 'plot_histogram',
                           'plot_heat_map', 'plot_sankey_diagram', 'plot_simple_sunburst_graph']
            }
        }

    @staticmethod
    def plot_bar_chart(x, y, data, title="Bar Chart"):
        fig = px.bar(data, x=x, y=y, title=title)
        return {'status': 'success', 'figure': fig}

    @staticmethod
    def plot_pie_chart(names, values, data, title="Pie Chart"):
        fig = px.pie(data, names=names, values=values, title=title)
        return {'status': 'success', 'figure': fig}

    @staticmethod
    def plot_histogram(x, data, title="Histogram"):
        fig = px.histogram(data, x=x, title=title)
        return {'status': 'success', 'figure': fig}

    @staticmethod
    def plot_heat_map(values, index, columns, aggregade_method='mean', title="Heatmap", data=None):
        pivot = data.pivot_table(values=values, index=index, columns=columns, aggfunc=aggregade_method)
        fig = px.imshow(pivot, title=title)
        return {'status': 'success', 'figure': fig}

    @staticmethod
    def plot_sankey_diagram(source_column, target_column, values, data):
        # Simplified Sankey
        fig = go.Figure(data=[go.Sankey(
            node = dict(label=["Source", "Target"]),
            link = dict(source=[0], target=[1], value=[data[values].sum()])
        )])
        fig.update_layout(title="Sankey Diagram")
        return {'status': 'success', 'figure': fig}

    @staticmethod
    def plot_simple_sunburst_graph(path, values, data, title="Sunburst"):
        fig = px.sunburst(data, path=path, values=values, title=title)
        return {'status': 'success', 'figure': fig}

    @staticmethod
    def display_image(result):
        if 'figure' in result:
            result['figure'].show()
        else:
            print("No figure to display.")

In [ ]:
class DataInspector:
    def __init__(self):
        self.df = None
        self.original_df = None

    def upload_data(self):
        print("Please upload your CSV file:")
        uploaded = files.upload()
        if uploaded:
            filename = list(uploaded.keys())[0]
            self.df = pd.read_csv(filename)
            self.original_df = self.df.copy()
            print(f"✅ Loaded dataset with shape: {self.df.shape}")
            return self.df
        return None

    def get_summary(self):
        """Comprehensive data summary."""
        print(f"\n{'='*60}")
        print("DATA SUMMARY")
        print(f"{'='*60}")
        print(f"Shape: {self.df.shape[0]:,} rows, {self.df.shape[1]} columns")
        print("\nNumerical columns:", self.df.select_dtypes(include=np.number).columns.tolist())
        print("Categorical columns:", self.df.select_dtypes(include='object').columns.tolist())
        display(self.df.head(10))
        print("\nMissing Values:")
        print(self.df.isnull().sum()[self.df.isnull().sum() > 0])

    def sanitize_garbage_strings(self):
        garbage = ['?', 'n/a', 'N/A', 'null', 'NULL', 'NaN', ' ', '']
        self.df.replace(garbage, np.nan, inplace=True)
        print("✅ Garbage strings converted to NaN.")

    def auto_correct_types(self):
        for col in self.df.columns:
            if self.df[col].dtype == 'object':
                try:
                    converted = pd.to_numeric(self.df[col], errors='coerce')
                    if not converted.isna().all():
                        self.df[col] = converted
                except:
                    pass

    def column_details(self):
        print("\nColumn Information:")
        for col in self.df.columns:
            print(f"{col}: {self.df[col].dtype} | Unique: {self.df[col].nunique()} | Missing: {self.df[col].isnull().sum()}")

    def get_categorical_summary(self):
        cat_cols = self.df.select_dtypes(include='object').columns
        for col in cat_cols:
            print(f"\n{col} value counts:")
            print(self.df[col].value_counts().head(10))

    def show_missing_data(self):
        missing = self.df.isnull().sum()
        missing = missing[missing > 0]
        if not missing.empty:
            print("Missing Data:")
            display(missing)
        else:
            print("No missing values found.")

    def handle_missing_values(self, strategy='median', columns=None):
        if columns is None:
            columns = self.df.columns
        for col in columns:
            if self.df[col].isnull().sum() > 0:
                if strategy == 'mean' and pd.api.types.is_numeric_dtype(self.df[col]):
                    self.df[col].fillna(self.df[col].mean(), inplace=True)
                elif strategy == 'median' and pd.api.types.is_numeric_dtype(self.df[col]):
                    self.df[col].fillna(self.df[col].median(), inplace=True)
                else:
                    self.df[col].fillna(self.df[col].mode()[0] if not self.df[col].mode().empty else np.nan, inplace=True)
        print(f"✅ Missing values handled with {strategy} strategy.")

    def handle_outliers(self, columns=None, find_and_delete=True, iqr_multiplier=1.5):
        if columns is None:
            columns = self.df.select_dtypes(include=np.number).columns.tolist()
        outlier_indices = set()
        for col in columns:
            if pd.api.types.is_numeric_dtype(self.df[col]):
                Q1 = self.df[col].quantile(0.25)
                Q3 = self.df[col].quantile(0.75)
                IQR = Q3 - Q1
                lower = Q1 - iqr_multiplier * IQR
                upper = Q3 + iqr_multiplier * IQR
                mask = (self.df[col] < lower) | (self.df[col] > upper)
                outlier_indices.update(self.df[mask].index)
        if find_and_delete:
            self.df = self.df.drop(index=list(outlier_indices))
            print(f"✅ Removed {len(outlier_indices)} outlier rows.")
        else:
            print(f"Found {len(outlier_indices)} potential outliers.")

    def delete_rows(self):
        print("Current row indices:", self.df.index.tolist()[:10], "...")
        indices = input("Enter indices to delete (comma separated): ")
        if indices:
            idx = [int(i.strip()) for i in indices.split(',')]
            self.df = self.df.drop(index=idx, errors='ignore')
            print("Rows deleted.")

    def delete_columns(self):
        print("Current columns:", self.df.columns.tolist())
        cols = input("Enter columns to delete (comma separated): ")
        if cols:
            col_list = [c.strip() for c in cols.split(',')]
            self.df = self.df.drop(columns=col_list, errors='ignore')
            print("Columns deleted.")

    def plot_numerical(self, column_names=None):
        if column_names is None:
            column_names = self.df.select_dtypes(include=np.number).columns.tolist()[:5]
        for col in column_names:
            if col in self.df.columns:
                self.plot_univariate_numeric(col)

    def plot_categorical(self, column_names=None):
        if column_names is None:
            column_names = self.df.select_dtypes(include='object').columns.tolist()[:4]
        for col in column_names:
            if col in self.df.columns:
                self.plot_categorical_frequency(col)

    def plot_univariate_numeric(self, column):
        data = self.df[column].dropna()
        fig = make_subplots(rows=1, cols=3, subplot_titles=("Violin", "Scatter", "Histogram"))
        fig.add_trace(go.Violin(y=data, box_visible=True), row=1, col=1)
        fig.add_trace(go.Scatter(x=list(range(len(data))), y=data, mode='markers'), row=1, col=2)
        fig.add_trace(go.Histogram(x=data), row=1, col=3)
        fig.update_layout(title=f"{column} Analysis", height=450)
        fig.show()

    def plot_categorical_frequency(self, column):
        html = PlottingMethods.plot_bar_chart(x=column, y=column, data=self.df, title=f"{column} Frequency")
        PlottingMethods.display_image(html)  # Simplified

    def plot_relationship(self, col1, col2):
        if pd.api.types.is_numeric_dtype(self.df[col1]) and pd.api.types.is_numeric_dtype(self.df[col2]):
            fig = px.scatter(self.df, x=col1, y=col2, trendline="ols")
        elif pd.api.types.is_numeric_dtype(self.df[col1]) or pd.api.types.is_numeric_dtype(self.df[col2]):
            cat = col1 if not pd.api.types.is_numeric_dtype(self.df[col1]) else col2
            num = col2 if not pd.api.types.is_numeric_dtype(self.df[col1]) else col1
            fig = px.box(self.df, x=cat, y=num)
        else:
            fig = px.bar(self.df.groupby(col1)[col2].value_counts().unstack())
        fig.show()

    def plot_numerical_correlation(self):
        num_df = self.df.select_dtypes(include=np.number)
        fig = px.imshow(num_df.corr(), text_auto=True, color_continuous_scale='RdBu_r')
        fig.show()

    def plot_categorical_correlation(self):
        print("Categorical correlation via Cramér's V not fully implemented in this demo.")

    def plot_all_associations_heatmap(self):
        # Basic numeric correlation for demo
        num_df = self.df.select_dtypes(include=np.number)
        corr = num_df.corr()
        fig = px.imshow(corr, text_auto=True, aspect="auto", color_continuous_scale='RdBu_r',
                       title="Association Heatmap (Numeric)")
        fig.show()

    def get_prepared_dataset(self, num_method='minmax', cat_method='onehot'):
        # Implementation from previous version
        num = self.extract_normalized_numeric_data(num_method)
        cat = self.extract_normalized_categorical_data(cat_method)
        return pd.concat([num, cat], axis=1)

    # Add missing helper methods
    def extract_normalized_numeric_data(self, method='minmax'):
        num_df = self.df.select_dtypes(include=np.number)
        if method == 'minmax':
            return (num_df - num_df.min()) / (num_df.max() - num_df.min())
        return num_df

## Quick Start Example

In [ ]:
# Initialize the inspector
inspector = DataInspector()

## Data Import

In [ ]:
# Option 1: Upload file
# inspector.upload_data()

# Option 2: Load Titanic directly
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
inspector.df = pd.read_csv(url)
print("Titanic dataset loaded successfully!")
print(inspector.df.shape)

## Initial Data Inspection and Cleaning

In [ ]:
inspector.sanitize_garbage_strings()
inspector.auto_correct_types()
inspector.get_summary()
inspector.column_details()
inspector.get_categorical_summary()
inspector.show_missing_data()

In [ ]:
# Handle missing values
inspector.handle_missing_values(strategy='median')

# Remove duplicates
inspector.df.drop_duplicates(inplace=True)

# Outlier handling
numerical_cols = ['Age', 'Fare', 'SibSp', 'Parch']
inspector.handle_outliers(columns=numerical_cols, find_and_delete=True)

## Visualizations

In [ ]:
inspector.plot_numerical(column_names=['Age', 'Fare'])
inspector.plot_categorical(column_names=['Sex', 'Pclass', 'Embarked'])
inspector.plot_relationship('Pclass', 'Fare')
inspector.plot_all_associations_heatmap()

## Custom Plotting with PlottingMethods

In [ ]:
PLT = PlottingMethods()

# Example plots
result = PLT.plot_bar_chart(x='Sex', y='Survived', data=inspector.df, title='Survival by Gender')
PLT.display_image(result)

result = PLT.plot_pie_chart(names='Pclass', values='Survived', data=inspector.df.groupby('Pclass')['Survived'].mean().reset_index(), title='Survival Rate by Class')
PLT.display_image(result)

## Prepared Dataset for Modeling

In [ ]:
prepared_df = inspector.get_prepared_dataset()
print("Prepared dataset shape:", prepared_df.shape)
display(prepared_df.head())